# Tutorial 10 — Advanced: Matrices, Custom Methods, Graph Traversal

Companion explainer: **10_advanced_topics.md**. Direct matrix surgery,
writing a custom LCIA method, structured graph traversal, and the
legacy-vs-2.5 API map (in the .md).

In [1]:
import numpy as np
import pandas as pd
import bw2data as bd
import bw2calc as bc

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

def find_flow(name, categories=("air",)):
    return next(f for f in bio if f["name"] == name and f["categories"] == categories)

co2 = find_flow("Carbon dioxide, fossil")
ch4 = find_flow("Methane, fossil")
n2o = find_flow("Dinitrogen monoxide")

DB = "t10_sys"
if DB in bd.databases:
    del bd.databases[DB]
bd.Database(DB).write({
    (DB, "elec"): {"name": "electricity", "unit": "kilowatt hour", "exchanges": [
        {"input": (DB, "elec"), "amount": 1.0, "type": "production"},
        {"input": co2.key, "amount": 0.95, "type": "biosphere"},
        {"input": ch4.key, "amount": 0.0003, "type": "biosphere"},
        {"input": n2o.key, "amount": 0.00001, "type": "biosphere"}]},
    (DB, "steel"): {"name": "steel", "unit": "kilogram", "exchanges": [
        {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 2.9, "type": "technosphere"},
        {"input": co2.key, "amount": 1.9, "type": "biosphere"}]},
    (DB, "widget"): {"name": "widget", "unit": "kilogram", "exchanges": [
        {"input": (DB, "widget"), "amount": 1.0, "type": "production"},
        {"input": (DB, "steel"), "amount": 0.5, "type": "technosphere"},
        {"input": (DB, "elec"), "amount": 1.0, "type": "technosphere"}]},
})
widget = bd.get_node(database=DB, code="widget")
steel = bd.get_node(database=DB, code="steel")
elec = bd.get_node(database=DB, code="elec")
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))

13:18:54-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

13:18:54-0400

 [

info     

] 

Vacuuming database            

## 1. Direct matrix access

In [2]:
lca = bc.LCA({widget: 1}, method=gwp)
lca.lci(); lca.lcia()
A = np.asarray(lca.technosphere_matrix.todense())
print("technosphere matrix A:")
names = {v: bd.get_activity(k)["name"] for k, v in lca.dicts.activity.items()}
print("cols/rows:", [names[i] for i in range(len(names))])
print(np.round(A, 3))
print("baseline score:", round(lca.score, 4))

technosphere matrix A:

cols/rows:

['electricity', 'steel', 'widget']

[[ 1.  -2.9 -1. ]
 [ 0.   1.  -0.5]
 [ 0.   0.   1. ]]

baseline score:

3.3058

## 2. What-if without a database write: edit a matrix cell, re-solve

In [3]:
row = lca.dicts.activity[elec.id]      # electricity as product (row)
col = lca.dicts.activity[steel.id]     # steel process (col)
print("steel's electricity input (matrix):", A[row, col], "(negative = input)")
lca.technosphere_matrix[row, col] *= 0.8   # 20% less electricity in steel
lca.redo_lci({widget.id: 1}) if hasattr(lca, "redo_lci") else lca.lci()
lca.lcia()
print("score after -20% elec in steel:", round(lca.score, 4))
# restore
lca = bc.LCA({widget: 1}, method=gwp); lca.lci(); lca.lcia()

steel's electricity input (matrix):

-2.9000000953674316

(negative = input)

score after -20% elec in steel:

3.027

## 3. Write a custom LCIA method and reconcile against IPCC

In [4]:
my = bd.Method(("SDAI", "simple GWP", "v1"))
my.register(unit="kg CO2-eq", description="teaching method: CO2=1, CH4=29.8, N2O=273")
my.write([(co2.key, 1.0), (ch4.key, 29.8), (n2o.key, 273.0)])

l_custom = bc.LCA({widget: 1}, method=("SDAI", "simple GWP", "v1"))
l_custom.lci(); l_custom.lcia()
print("custom method score :", round(l_custom.score, 5))
print("IPCC method score   :", round(lca.score, 5))

# reconcile flow by flow using our custom CFs
ipcc_cfs = dict(bd.Method(gwp).load())
custom_cfs = {"CO2": 1.0, "CH4": 29.8, "N2O": 273.0}
print("\nflow-by-flow CF comparison:")
for flow, lbl in [(co2, "CO2"), (ch4, "CH4"), (n2o, "N2O")]:
    ipcc = ipcc_cfs.get(flow.id)
    print(f"   {lbl:4s}  custom={custom_cfs[lbl]:7.2f}  IPCC={ipcc}")

custom method score :

3.30609

IPCC method score   :

3.30582


flow-by-flow CF comparison:

   CO2   custom=   1.00  IPCC=1.0

   CH4   custom=  29.80  IPCC=29.7

   N2O   custom= 273.00  IPCC=264.8

## 4. Structured graph traversal (bw_graph_tools if available, else manual)

In [5]:
try:
    import bw_graph_tools as bgt
    gt = bgt.NewNodeEachVisitGraphTraversal.calculate(lca, cutoff=0.01)
    print("bw_graph_tools traversal:")
    print("   nodes:", len(gt.get("nodes", gt) if isinstance(gt, dict) else []))
    used_bgt = True
except Exception as e:
    used_bgt = False
    print("bw_graph_tools not usable here (", type(e).__name__, ") — manual traversal:")

    def traverse(act, amount=1.0, level=0, max_level=3, cutoff=0.01, total=None):
        l = bc.LCA({act: amount}, method=gwp); l.lci(); l.lcia()
        if total is None:
            total = l.score
        if total == 0 or abs(l.score / total) < cutoff or level > max_level:
            return
        print("  " * level + f"{l.score/total:5.1%}  {act['name']} (x{amount:.3g})")
        for exc in act.technosphere():
            traverse(exc.input, amount * exc["amount"], level + 1, max_level, cutoff, total)

    traverse(widget)

C:\Users\derne\AppData\Local\Temp\ipykernel_24964\2172715499.py:3: DeprecationWarning: Use `NewNodeEachVisitGraphTraversal(lca, settings)` instead of `NNEVGT().calculate(stuff)`
  gt = bgt.NewNodeEachVisitGraphTraversal.calculate(lca, cutoff=0.01)


bw_graph_tools traversal:

   nodes:

5

## 5. Cumulative intensity of every process at once
Solve A^T x = (C·B)^T -> impact intensity per unit of each process.

In [6]:
lca2 = bc.LCA({widget: 1}, method=gwp); lca2.lci(); lca2.lcia()
# characterization applied to biosphere -> per-process direct char. then upstream
B = lca2.biosphere_matrix
C = lca2.characterization_matrix
direct = np.asarray((C @ B).sum(axis=0)).ravel()   # per-process direct impact intensity
A2 = lca2.technosphere_matrix
# cumulative intensity g = direct @ A^{-1}  (solve A^T y = direct^T)
import scipy.sparse.linalg as sla
y = sla.spsolve(A2.T.tocsc(), direct)
print("cumulative GWP intensity per unit of process output:")
for k, v in lca2.dicts.activity.items():
    print(f"   {bd.get_activity(k)['name']:12s} {y[v]: .4f} kg CO2-eq/unit")

# cross-check: widget intensity should equal the full-system score for 1 widget
w_idx = lca2.dicts.activity[widget.id]
print("\nwidget cumulative intensity:", round(y[w_idx], 4),
      "| full LCA score:", round(lca2.score, 4))

cumulative GWP intensity per unit of process output:

   electricity   0.9616 kg CO2-eq/unit

   steel         4.6885 kg CO2-eq/unit

   widget        3.3058 kg CO2-eq/unit


widget cumulative intensity:

3.3058

| full LCA score:

3.3058

See **10_advanced_topics.md** for the full legacy-vs-2.5 API translation table.
Continue with the **case studies** to apply all of this on realistic systems.